# Meta DE Onsite — 5 Schemas in 5 Schemas (Notebook)

The 5 most-asked 2026 Meta DE data-modeling questions, each
exercised against the worked schema in `code/meta_onsite_schemas.sql`.

Pairs with `design/04_concrete_solutions.md` (the worked answers) and
`design/05_companies_to_research.md` (which Meta org each surface belongs to).

Order: Reels (Q1) -> cross-platform (Q2) -> Ads Auction (Q3) -> ride-share (Q4) -> metric drop (Q5).

In [1]:
import os, sqlite3
BASE = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
schema_path = os.path.join(BASE, 'code', 'meta_onsite_schemas.sql')
conn = sqlite3.connect(':memory:')
with open(schema_path) as f:
    conn.executescript(f.read())
print('5 schemas loaded')

5 schemas loaded


## Q1 — Reels star schema (the SCD2 algorithm_version dim)

Show the current algorithm version, then count fact rows and bridge rows.

In [2]:
for row in conn.execute("""
    SELECT algorithm_name, valid_from, valid_to, is_current
    FROM   dim_algorithm_version
    ORDER BY algorithm_version_id
"""):
    print(row)
print()
for row in conn.execute("""
    SELECT 'fct_reel_view' AS tbl, COUNT(*) AS n FROM fct_reel_view
    UNION ALL
    SELECT 'reel_hashtag_bridge', COUNT(*) FROM reel_hashtag_bridge
    UNION ALL
    SELECT 'reel_audio_bridge',   COUNT(*) FROM reel_audio_bridge
"""):
    print(row)

('reels_v1', '2026-01-01', '2026-01-15', 0)
('reels_v2', '2026-01-15', '2026-02-01', 0)
('reels_v3', '2026-02-01', '9999-12-31', 1)

('fct_reel_view', 6)
('reel_hashtag_bridge', 5)
('reel_audio_bridge', 3)


## Q2 — Cross-platform identity resolution (the bridge table)

User 1 maps to IG + WA. User 2 is FB-only. User 3 is IG-only.

In [3]:
for row in conn.execute("""
    SELECT unified_user_id, platform, platform_user_id
    FROM   user_identity_bridge
    ORDER BY unified_user_id, platform
"""):
    print(row)

(1, 'IG', 101)
(1, 'WA', 201)
(2, 'FB', 102)
(3, 'IG', 103)


## Q3 — Ads Auction: time-travel (immutable fact) + current-state (SCD2 dim)

5 auctions: 3 won, 2 lost. 3 impressions on the 3 won auctions. Total revenue = 730 cents.

In [4]:
for row in conn.execute("""
    SELECT won_flag, COUNT(*) AS n
    FROM   fct_auction_event
    GROUP BY won_flag
    ORDER BY won_flag
"""):
    print(row)
print()
for row in conn.execute("""
    SELECT ad_id, COUNT(*) AS n, SUM(revenue_cents) AS rev
    FROM   fct_impression
    GROUP BY ad_id
    ORDER BY ad_id
"""):
    print(row)

(0, 2)
(1, 3)

(1, 2, 550)
(2, 1, 180)


## Q5 — Metric drop: localize the segment

The like_rate for segment 1 (US, mobile) dropped from 0.42 to 0.30 on Feb 14.
Segment 2 (US, desktop) is stable around 0.40. The drop is **localized**.

In [5]:
for row in conn.execute("""
    SELECT time_bucket, segment_id, value, sample_size
    FROM   fct_metric_value
    WHERE  metric_id = 1
    ORDER BY segment_id, time_bucket
"""):
    print(row)

('2026-02-12', 1, 0.42, 1000000)
('2026-02-13', 1, 0.41, 1050000)
('2026-02-14', 1, 0.3, 1100000)
('2026-02-15', 1, 0.28, 1080000)
('2026-02-12', 2, 0.4, 200000)
('2026-02-13', 2, 0.41, 210000)
('2026-02-14', 2, 0.4, 220000)
('2026-02-15', 2, 0.41, 215000)
